# Nanoclusters of Gold

## 0. Introduction

This notebook demonstrates creation of Gold nanoclusters with cuboctahedral and icosahedral shapes. The notebook is based on the following publication:


> **A. H. Larsen, J. Kleis, K. S. Thygesen, J. K. Nørskov, and K. W. Jacobsen**,
> "Electronic shell structure and chemisorption on gold nanoparticles",
> *Phys. Rev. B 84, 245429 (2011)*,
> [DOI: 10.1103/PhysRevB.84.245429](https://doi.org/10.1103/PhysRevB.84.245429)
> 

In this notebook we will build the series between the closed shells used in Fig. 4 of the manuscript (N = 30–60) and reproduce materials from FIG. 2:

![Fig 2.](https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/0d_materials/nanocluster_gold/0-manuscript-image.webp
)


## 1. Prepare the Environment


### 1.1. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.2. Set up cluster parameters
Parameters for the cuboctahedron and icosahedron shapes can be found at [ASE Clusters](https://wiki.fysik.dtu.dk/ase/ase/cluster/cluster.html#ase.cluster.Cluster).

In [ ]:
from mat3ra.made.tools.build.pristine_structures.zero_dimensional.nanoparticle import NanoparticleShapesEnum
from types import SimpleNamespace

ELEMENT = "Au"  # Chemical symbol of the element to create nanoparticles from
SHAPES = ["icosahedron"]  # , "cuboctahedron"
CLUSTER_SIZES = [55]  # list(range(30, 61))
RANDOM_SEED = 42
VACUUM = 5.0  # Å on each side of the cluster
COORDINATION_CUTOFF = 3.5  # Å, distance within which atoms count as neighbors


def length(cutoff):
    # The parameters of to achieve cuboctahedron shape from the octahedron
    return 2 * cutoff + 1


NANOPARTICLE_CONFIGS = [
    SimpleNamespace(
        name="cuboctahedron",
        shape=NanoparticleShapesEnum.OCTAHEDRON,
        parameters=SimpleNamespace(
            length=length(2),
            cutoff=2,
        )
    ),
    SimpleNamespace(
        name="cuboctahedron",
        shape=NanoparticleShapesEnum.OCTAHEDRON,
        parameters=SimpleNamespace(
            length=length(3),
            cutoff=3,
        )
    ),
    SimpleNamespace(
        name="cuboctahedron",
        shape=NanoparticleShapesEnum.OCTAHEDRON,
        parameters=SimpleNamespace(
            length=length(4),
            cutoff=4,
        )
    ),
    SimpleNamespace(
        name="icosahedron",
        shape=NanoparticleShapesEnum.ICOSAHEDRON,
        parameters=SimpleNamespace(
            noshells=3,
        )
    ),
    SimpleNamespace(
        name="icosahedron",
        shape=NanoparticleShapesEnum.ICOSAHEDRON,
        parameters=SimpleNamespace(
            noshells=4,
        )
    ),
    SimpleNamespace(
        name="icosahedron",
        shape=NanoparticleShapesEnum.ICOSAHEDRON,
        parameters=SimpleNamespace(
            noshells=5,
        )
    )
]


### 1.3. Get input material

In [ ]:
from mat3ra.standata.materials import Materials
from mat3ra.made.material import Material

material = Material.create(Materials.get_by_name_first_match("Gold"))

## 2. Create the Target Material(s)

In [ ]:
import numpy as np
from ase.neighborlist import neighbor_list
from mat3ra.made.tools.convert import to_ase
from mat3ra.made.tools.helpers import create_nanoparticle_by_shape
from mat3ra.made.tools.modify import add_vacuum, add_vacuum_sides, filter_by_ids, remove_vacuum

nanoparticles = [create_nanoparticle_by_shape(
    crystal=material,
    shape=config.shape,
    parameters=config.parameters
) for config in NANOPARTICLE_CONFIGS]


def strip_to_size(cluster, size):
    random_generator = np.random.default_rng(RANDOM_SEED)
    atoms = to_ase(cluster)
    parent_coordination = np.bincount(neighbor_list("i", atoms, COORDINATION_CUTOFF))
    in_smaller_shell = parent_coordination == parent_coordination.max()
    height_along_111 = atoms.positions.sum(axis=1)
    on_kept_facet = np.isclose(height_along_111, height_along_111.max())
    remaining = np.arange(len(atoms))
    while len(remaining) > size:
        coordination = np.bincount(neighbor_list("i", atoms[remaining], COORDINATION_CUTOFF))
        removable = ~in_smaller_shell[remaining] & ~on_kept_facet[remaining]
        if not removable.any():
            removable = ~in_smaller_shell[remaining]
        lowest_coordinated = np.flatnonzero(removable & (coordination == coordination[removable].min()))
        remaining = np.delete(remaining, random_generator.choice(lowest_coordinated))
    return filter_by_ids(cluster, remaining.tolist())


nanoparticle_series = []
for shape in SHAPES:
    closed_shells = [
        nanoparticle for config, nanoparticle in zip(NANOPARTICLE_CONFIGS, nanoparticles) if config.name == shape
    ]
    for size in CLUSTER_SIZES:
        closed_shell = next(shell for shell in closed_shells if len(shell.basis.elements.ids) >= size)
        cluster = remove_vacuum(strip_to_size(closed_shell, size), fixed_padding=0)
        cluster = add_vacuum(cluster, VACUUM, on_top=True, to_bottom=True)
        cluster = add_vacuum_sides(cluster, VACUUM, on_x=True, on_y=True)
        cluster.name = f"{ELEMENT}{size} {shape.capitalize()}"
        nanoparticle_series.append(cluster)

## 3. Visualize the Result(s)

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

cuboctahedra_particles = nanoparticles[0:3]
icosahedra_particles = nanoparticles[3:6]
visualize(cuboctahedra_particles, rotation="45y,45x")
visualize(icosahedra_particles, rotation="60x")
visualize(nanoparticle_series, rotation="60x")

## 4. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials(nanoparticles + nanoparticle_series)
for idx, nanoparticle in enumerate(nanoparticles + nanoparticle_series):
    download_content_to_file(nanoparticle.to_json(), f"gold_nanocluster_{idx}.json")